# Clase 9 — Archivos, SQL básico y prompts estructurados

En la Clase 8 convertimos bloques sueltos en funciones reutilizables.  
Ahora vamos a hacer el siguiente puente: tomar datos desde archivos, consultarlos con SQL y convertirlos en mensajes listos para un modelo de chat.

Este notebook no usa todavía `llama.cpp`.  
La idea es preparar la lógica de entrada y salida que en la **Clase 10** se conectará con un modelo local real.

**Objetivos de la clase:**
- Leer registros desde `csv` y `json`.
- Crear una base SQLite en memoria y consultarla con `SELECT`, `WHERE`, `ORDER BY`, `LIMIT`.
- Transformar resultados en mensajes `[{role, content}, ...]`.
- Configurar parámetros de generación con lógica simple.
- Simular un chat antes de pasar al modelo real.


---
## 1. Archivos: la misma información puede venir en formatos distintos

Dos formatos muy comunes para datos estructurados son:
- **CSV**: tabla simple, buena para listas de registros.
- **JSON**: más flexible, muy usado para APIs y configuraciones.

En esta clase vamos a usar solo librerías estándar de Python:
- `csv`
- `json`
- `pathlib`
- `tempfile`


In [ ]:
import csv
import json
import tempfile
from pathlib import Path


registros_demo = [
    {
        "nombre": "manzana roja",
        "clase": "manzana",
        "color": "rojo",
        "calorias": 95,
        "descripcion": "fruta crocante y dulce, útil para snacks breves",
    },
    {
        "nombre": "manzana verde",
        "clase": "manzana",
        "color": "verde",
        "calorias": 80,
        "descripcion": "más ácida, suele usarse en preparaciones frescas",
    },
    {
        "nombre": "naranja de mesa",
        "clase": "naranja",
        "color": "naranja",
        "calorias": 62,
        "descripcion": "cítrica y jugosa, común en desayunos",
    },
    {
        "nombre": "limon amarillo",
        "clase": "limon",
        "color": "amarillo",
        "calorias": 29,
        "descripcion": "muy ácido, útil para saborizar y condimentar",
    },
]

carpeta_demo = Path(tempfile.gettempdir()) / "modulo_3_clase_9"
carpeta_demo.mkdir(exist_ok=True)

csv_path = carpeta_demo / "frutas_demo.csv"
json_path = carpeta_demo / "frutas_demo.json"

with csv_path.open("w", newline="", encoding="utf-8") as archivo_csv:
    writer = csv.DictWriter(archivo_csv, fieldnames=registros_demo[0].keys())
    writer.writeheader()
    writer.writerows(registros_demo)

with json_path.open("w", encoding="utf-8") as archivo_json:
    json.dump(registros_demo, archivo_json, ensure_ascii=False, indent=2)


def cargar_registros(ruta):
    ruta = Path(ruta)

    if ruta.suffix == ".csv":
        with ruta.open("r", encoding="utf-8", newline="") as archivo:
            return list(csv.DictReader(archivo))

    if ruta.suffix == ".json":
        with ruta.open("r", encoding="utf-8") as archivo:
            return json.load(archivo)

    raise ValueError(f"Formato no soportado: {ruta.suffix}")


registros_csv = cargar_registros(csv_path)
registros_json = cargar_registros(json_path)

print("CSV :", csv_path)
print("JSON:", json_path)
print("Primer registro desde CSV :", registros_csv[0])
print("Primer registro desde JSON:", registros_json[0])


---
## 2. SQLite en memoria: consultar datos con SQL

SQL sirve para hacer preguntas estructuradas sobre una tabla:
- `SELECT` → qué columnas quiero,
- `WHERE` → qué filtro aplico,
- `ORDER BY` → en qué orden quiero los resultados,
- `LIMIT` → cuántas filas necesito.

Vamos a usar `sqlite3` porque viene con Python y permite crear una base **en memoria** para practicar sin instalar nada extra.


In [ ]:
import sqlite3


def crear_base_demo(conn, registros):
    conn.execute("DROP TABLE IF EXISTS frutas")
    conn.execute(
        '''
        CREATE TABLE frutas (
            nombre TEXT,
            clase TEXT,
            color TEXT,
            calorias INTEGER,
            descripcion TEXT
        )
        '''
    )

    filas = []
    for registro in registros:
        filas.append(
            (
                registro["nombre"],
                registro["clase"],
                registro["color"],
                int(registro["calorias"]),
                registro["descripcion"],
            )
        )

    conn.executemany("INSERT INTO frutas VALUES (?, ?, ?, ?, ?)", filas)
    conn.commit()


def consultar_contexto(conn, criterio):
    cursor = conn.execute(
        '''
        SELECT nombre, clase, color, calorias, descripcion
        FROM frutas
        WHERE clase = ? OR color = ?
        ORDER BY calorias DESC, nombre ASC
        LIMIT 3
        ''',
        (criterio, criterio),
    )
    return cursor.fetchall()


In [ ]:
conn = sqlite3.connect(":memory:")
crear_base_demo(conn, registros_csv)

contexto_manzana = consultar_contexto(conn, "manzana")
print("consultar_contexto('manzana'):", contexto_manzana)

top_calorias = conn.execute(
    '''
    SELECT nombre, calorias
    FROM frutas
    ORDER BY calorias DESC
    LIMIT 3
    '''
).fetchall()

print("Top 3 por calorías:", top_calorias)


---
## 3. Del resultado SQL al prompt: mensajes con `role` y `content`

Un modelo de chat no recibe una tabla SQL.  
Recibe una lista de mensajes.

Estructura base:
```python
[
    {"role": "system", "content": "..."},
    {"role": "user",   "content": "..."},
]
```

El contexto recuperado de la base puede transformarse en texto y pegarse dentro del mensaje del usuario.


In [ ]:
def armar_mensajes(system_prompt, user_prompt, contexto):
    lineas_contexto = []

    for nombre, clase, color, calorias, descripcion in contexto:
        lineas_contexto.append(
            f"- {nombre} ({clase}, {color}, {calorias} kcal): {descripcion}"
        )

    if lineas_contexto:
        bloque_contexto = "\n".join(lineas_contexto)
    else:
        bloque_contexto = "- Sin contexto adicional disponible."

    return [
        {"role": "system", "content": system_prompt.strip()},
        {
            "role": "user",
            "content": (
                f"Contexto disponible:\n{bloque_contexto}\n\n"
                f"Pregunta:\n{user_prompt.strip()}"
            ),
        },
    ]


mensajes = armar_mensajes(
    "Eres un asistente claro y breve.",
    "Compará manzana y naranja en 3 frases.",
    consultar_contexto(conn, "naranja"),
)

for mensaje in mensajes:
    print(mensaje["role"].upper())
    print(mensaje["content"])
    print("-" * 60)


---
## 4. Configurar generación antes de hablar con el modelo

Antes de enviar un pedido a un chat local conviene decidir:
- si la tarea requiere precisión o creatividad,
- cuánto texto esperamos,
- si vale la pena usar streaming.

Esa decisión todavía no ejecuta nada.  
Solo construye una **configuración** coherente para la consulta.


In [ ]:
def configurar_generacion(tipo_tarea, largo_esperado):
    if tipo_tarea in {"resumir", "codigo", "clasificar"}:
        temperature = 0.2
    elif tipo_tarea in {"ideas", "creativo"}:
        temperature = 0.8
    else:
        temperature = 0.5

    max_tokens = min(max(largo_esperado, 60), 220)
    stream = largo_esperado > 100

    return {
        "temperature": temperature,
        "max_tokens": max_tokens,
        "stream": stream,
    }


print(configurar_generacion("resumir", 80))
print(configurar_generacion("ideas", 140))


---
## 5. Simular el chat sin usar todavía `llama.cpp`

En la próxima clase el modelo va a existir de verdad.  
Hoy nos alcanza con una simulación para verificar que:
- los mensajes tienen el formato correcto,
- la configuración tiene sentido,
- y el flujo general ya está armado.


In [ ]:
def simular_chat(mensajes, config):
    ultima_pregunta = ""

    for mensaje in reversed(mensajes):
        if mensaje["role"] == "user":
            ultima_pregunta = mensaje["content"]
            break

    fragmento = ultima_pregunta.split("Pregunta:")[-1].strip().splitlines()[0]
    modo = "streaming" if config["stream"] else "respuesta completa"

    return (
        "[SIMULACION]\n"
        f"- modo: {modo}\n"
        f"- temperature: {config['temperature']}\n"
        f"- max_tokens: {config['max_tokens']}\n"
        f"- respuesta tentativa: Respondería a '{fragmento[:80]}' usando el contexto recuperado."
    )


config_demo = configurar_generacion("resumir", 120)
print(simular_chat(mensajes, config_demo))


---
## 6. Decisiones de control antes de enviar el prompt

Igual que en la Clase 4, antes de ejecutar algo podemos preguntar:
- ¿el prompt está vacío?
- ¿es demasiado corto?
- ¿es demasiado largo?
- ¿conviene usar temperatura baja?
- ¿hace falta streaming?

Esa lógica será muy útil en la Clase 10 cuando el modelo ya esté cargado.


In [ ]:
system_prompt = "Eres un asistente claro y preciso."
user_prompt = "Compará manzana y naranja en 3 frases claras."
tipo_tarea = "resumir"
largo_esperado = 120

prompt_listo = user_prompt.strip() != "" and len(user_prompt) >= 15
prompt_muy_largo = len(user_prompt) > 160
usar_stream = largo_esperado > 100
usar_temperatura_baja = tipo_tarea in {"resumir", "codigo", "clasificar"}

print("¿Prompt listo?            ", prompt_listo)
print("¿Prompt demasiado largo?  ", prompt_muy_largo)
print("¿Usar streaming?          ", usar_stream)
print("¿Usar temperatura baja?   ", usar_temperatura_baja)

if not prompt_listo:
    decision = "pedir aclaracion"
elif prompt_muy_largo:
    decision = "acortar prompt"
else:
    decision = "enviar"

print("Decisión final:", decision)


---
## 📝 Actividad 1 — Cargar datos y consultar los 3 registros más relevantes

**Consigna:**
- Crear una base en memoria con `crear_base_demo(...)`.
- Ejecutar una consulta SQL con `SELECT`, `WHERE`, `ORDER BY` y `LIMIT`.
- Mostrar las 3 filas más relevantes.

Podés usar el criterio que prefieras: calorías, color o clase.


In [ ]:
# ACTIVIDAD 1: cargar datos y consultar los 3 registros más relevantes
conn_actividad = sqlite3.connect(":memory:")
crear_base_demo(conn_actividad, registros_json)

consulta_sql = '''
SELECT nombre, clase, color, calorias
FROM frutas
WHERE calorias >= 60
ORDER BY calorias DESC
LIMIT 3
'''

# TODO: si querés, modificá la consulta para trabajar con otro criterio
filas = conn_actividad.execute(consulta_sql).fetchall()

print(filas)


---
### Actividad 2 — Armar mensajes con contexto recuperado

**Consigna:**
- Recuperá contexto con `consultar_contexto(...)`.
- Construí la lista de mensajes con `armar_mensajes(...)`.
- Verificá que quede con los roles correctos.


In [ ]:
# ACTIVIDAD 2: armar mensajes con contexto recuperado
contexto = consultar_contexto(conn, "manzana")
system_prompt = "Eres un asistente breve y ordenado."
user_prompt = "Explicá la diferencia entre manzana y naranja usando el contexto."

mensajes = [
    {"role": "system", "content": ""},
    {"role": "user", "content": ""},
]

# TODO: reemplazá la lista manual por una llamada a armar_mensajes(...)

print(mensajes)


---
### Actividad 3 — Decidir si el pedido está listo y simular el chat

**Consigna:**
- Si el prompt está vacío o es demasiado corto → `pedir aclaracion`
- Si es demasiado largo → `acortar prompt`
- Si está bien → `enviar`

Si la decisión final es `enviar`, ejecutá `simular_chat(...)`.


In [ ]:
# ACTIVIDAD 3: decidir si el pedido está listo y simular el chat
contexto = consultar_contexto(conn, "naranja")
system_prompt = "Eres un asistente de nutrición escolar."
user_prompt = "Compará la naranja y el limon en 3 frases."
tipo_tarea = "resumir"
largo_esperado = 140

decision = "pendiente"
config = configurar_generacion(tipo_tarea, largo_esperado)
mensajes = armar_mensajes(system_prompt, user_prompt, contexto)

# TODO:
# - si el prompt está vacío o es muy corto -> "pedir aclaracion"
# - si es demasiado largo -> "acortar prompt"
# - si está bien -> "enviar"

if decision == "enviar":
    print(simular_chat(mensajes, config))
else:
    print("Decisión actual:", decision)
    print("Config sugerida :", config)


---
## ✅ Resumen

| Pieza | Qué resuelve |
|---|---|
| `cargar_registros(...)` | Leer datos desde `csv` o `json` |
| `crear_base_demo(...)` | Crear una tabla SQLite en memoria |
| `consultar_contexto(...)` | Recuperar filas relevantes con SQL |
| `armar_mensajes(...)` | Convertir contexto en mensajes de chat |
| `configurar_generacion(...)` | Elegir parámetros coherentes |
| `simular_chat(...)` | Probar el flujo sin un modelo real |

**Lo que construiste hoy:**
- una mini cadena de trabajo desde archivo → SQL → contexto → mensajes → configuración.

**Lo que viene:**
- **Clase 10**: el formato de mensajes y la lógica de configuración se conectan por fin con un modelo local real usando `llama.cpp`.
